Γραμμάτης Βασίλης, 9144

Στρίκος Κωνσταντίνος, 9517

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical



def read_dataset():
  # we use header = None because pandas automatically thinks that we have headers at the first row
  dataset_train = pd.read_csv('datasetTV.csv', header = None)
  dataset_val = pd.read_csv('datasetTest.csv', header = None)

  # print('Dataset used for training has the following form:\n', dataset_train.describe())
  # dataset_train.describe().to_csv('datasetTV_describe.csv')

  # print('Dataset used for testing has the following form:\n', dataset_val.describe())
  # dataset_val.describe().to_csv('datasetTest_describe.csv')

  X = dataset_train.iloc[:, :-1].values
  y = dataset_train.iloc[:, -1].values - 1  # make labels [0, 4] to use them easier
  print ('y is:', y)

  X_val = dataset_val.values

  scaler = MinMaxScaler() # default feature_range=(0,1)
  X = scaler.fit_transform(X)
  X_val = scaler.transform(X_val) # scaler is already fitted

  X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                      test_size=0.3,
                                                      random_state=0,
                                                      stratify = y)
  return X_train, X_test, y_train, y_test, X_val



def build_model(X_train, y_train, dropout_prob, epochs, batch_size, validation_split, lr):
  model = Sequential([
      Input(shape=(X_train.shape[1],)),
      Dense(512, activation = 'relu'),
      BatchNormalization(),
      Dropout(dropout_prob),
      Dense(256, activation = 'relu'),
      BatchNormalization(),
      Dropout(dropout_prob),
      Dense(128, activation = 'relu'),
      BatchNormalization(),
      Dropout(dropout_prob),
      Dense(64, activation = 'relu'),
      BatchNormalization(),
      Dropout(dropout_prob),
      Dense(5, activation = 'softmax') # softmax for multiclass classification
  ])

  model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
                # sparce_categorical_crossentropy, because the labels are not one-hot encoded
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy'])

  return model


def get_callbacks():
  early_stopping = EarlyStopping(
      monitor = 'val_accuracy',
      patience = 10,
      restore_best_weights = True,
      start_from_epoch = 5
  )
  reduce_lr = ReduceLROnPlateau(
      monitor = 'val_loss',
      factor = 0.2,
      patience = 5
  )
  return [early_stopping, reduce_lr]


def plot_history(history):
  plt.figure(figsize = (8,5))
  plt.plot(history.history['loss'], label = 'train_loss')
  plt.plot(history.history['val_loss'], label = 'val_loss')
  plt.xlabel('Epochs')
  plt.ylabel('Loss')
  plt.legend()
  plt.grid(True)
  plt.title('Training and Validation Loss')
  plt.show()

  plt.figure(figsize = (8,5))
  plt.plot(history.history['accuracy'], label = 'train_accuracy')
  plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
  plt.xlabel('Epochs')
  plt.ylabel('Accuracy')
  plt.legend()
  plt.grid(True)
  plt.title('Training and Validation Accuracy')
  plt.show()



# def save_results_to_excel(results, hyperparameters, file_path = 'results.xlsx'):
#   try:
#       new_row = pd.DataFrame ({
#           'batch_size': hyperparameters['batch_size'],
#           'dropout_prob': hyperparameters['dropout_prob'],
#           'lr': hyperparameters['lr'],
#           'epochs': hyperparameters['epochs'],
#           'test_accuracy': results['test_accuracy']
#       }, index=[0])

#       if os.path.exists(file_path):
#           existing_df = pd.read_excel(file_path, sheet_name = 'Results')
#           updated_df = pd.concat([existing_df, new_row], ignore_index = True)

#           with pd.ExcelWriter(file_path, engine = 'openpyxl', mode = 'w') as writer:
#               updated_df.to_excel(writer, index = False, sheet_name='Results')
#       else:
#           new_row.to_excel(file_path, index = False, sheet_name = 'Results')

#       print(f'Results successfully saved to {file_path}')
#   except Exception as e:
#       print(f'Error saving results to {file_path}: {e}')


def main():
  np.random.seed(0)
  tf.random.set_seed(0)

  X_train, X_test, y_train, y_test, X_val = read_dataset()

  # hyperparams
  batch_size = 128
  dropout_prob = 0.3
  lr = 0.001
  epochs = 80

  print(f'Training model with batch_size = 128, dropout_prob = 0.3, lr = 0.001, epochs = 80')

  model = build_model(
      X_train, y_train,
      batch_size = batch_size,
      dropout_prob = dropout_prob,
      lr = lr,
      epochs = epochs,
      validation_split = 0.2)

  callbacks = get_callbacks()

  history = model.fit(
        X_train, y_train,
        epochs = epochs,
        batch_size = batch_size,
        validation_split = 0.2,
        callbacks = callbacks,
        verbose = 2
    )


  print('Evaluation on test set...')
  test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
  print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

  print('Making predictions on validation dataset...')
  predictions = model.predict(X_val)


  # convert probabilities to labels {0-4}
  predicted_labels = np.argmax(predictions, axis=1)
  # convert to {1-5}
  predicted_labels = predicted_labels + 1

  # print ('Predicted labels are: ', predicted_labels)

  # reshape the predicted labels to a column vector
  # predicted_labels_vertical = predicted_labels[np.newaxis, :].T
  # print ('Vertical predicted labels are:', predicted_labels_vertical)

  # save the predicted labels to a numpy array file
  np.save('labels41.npy', predicted_labels)
  print('Predicted labels saved to labels41.npy.')
  plot_history(history)
  print ('Process ended.')


if __name__ == '__main__':
  main()